# Phân loại sách bằng SVM và SMO

Bài toán là dự đoán sách thuộc danh mục **Fiction** hay **Nonfiction** dựa vào tiêu đề và mô tả. Dữ liệu lấy từ books_to_scrape.csv đã được cào từ LAB1

## 1. Đọc và chia dữ liệu

Chọn hai danh mục để tạo bài toán phân loại nhị phân. Giữ lại khoảng 20% số sách của mỗi danh mục để kiểm tra.

In [2]:
from collections import Counter

import numpy as np

from book_data import CATEGORY_LABELS, load_books, split_by_category
from main import DEFAULT_CSV
from smo import LinearSVM
from tfidf import TfidfVectorizer

books = load_books(DEFAULT_CSV)
train, test = split_by_category(books, seed=42)

print(f"Số sách thuộc hai danh mục: {len(books)}")
for split_name, split in [("Huấn luyện", train), ("Kiểm tra", test)]:
    counts = Counter(book.category for book in split)
    print(f"{split_name}: {len(split)} cuốn")
    for category in CATEGORY_LABELS:
        print(f"  {category}: {counts[category]} cuốn")

Số sách thuộc hai danh mục: 175
Huấn luyện: 140 cuốn
  Nonfiction: 88 cuốn
  Fiction: 52 cuốn
Kiểm tra: 35 cuốn
  Nonfiction: 22 cuốn
  Fiction: 13 cuốn


## 2. Tạo đặc trưng TF-IDF

Gộp tiêu đề và mô tả thành văn bản cho từng cuốn. Từ vựng và trọng số IDF được học trên tập huấn luyện, sau đó dùng chung để chuyển đổi cả hai tập.

In [3]:
train_text = [book.text for book in train]
test_text = [book.text for book in test]

vectorizer = TfidfVectorizer().fit(train_text)
train_features = vectorizer.transform(train_text)
test_features = vectorizer.transform(test_text)

train_labels = np.array([CATEGORY_LABELS[book.category] for book in train])
test_labels = np.array([CATEGORY_LABELS[book.category] for book in test])

print(f"Số từ trong từ vựng: {len(vectorizer.vocabulary)}")

Số từ trong từ vựng: 6519


## 3. Huấn luyện SVM bằng SMO

Dùng SVM tuyến tính với biên mềm và hệ số phạt $C=2$. Mỗi bước SMO cập nhật hai hệ số $\alpha$ sao cho tổng $\sum_i y_i\alpha_i$ vẫn bằng 0.

In [4]:
model = LinearSVM(c=2.0).fit(train_features, train_labels)

print(f"Số lần SMO cập nhật cặp: {model.updates}")

Số lần SMO cập nhật cặp: 298


## 4. Đánh giá trên tập kiểm tra

So sánh mô hình với cách luôn đoán danh mục xuất hiện nhiều nhất trong tập huấn luyện. Mỗi dòng trong bảng là nhãn thật của sách.

In [5]:
predicted = model.predict(test_features)
correct = np.count_nonzero(predicted == test_labels)
majority_label = Counter(train_labels).most_common(1)[0][0]
baseline = np.count_nonzero(test_labels == majority_label)

print(f"Độ chính xác: {correct}/{len(test)} ({correct / len(test):.1%})")
print(f"Luôn đoán lớp đông nhất: {baseline}/{len(test)} ({baseline / len(test):.1%})")
print()
print(f"{'Nhãn thật':<14} {'Đoán Nonfiction':>20} {'Đoán Fiction':>18}")
for category, label in CATEGORY_LABELS.items():
    counts = [
        np.count_nonzero((test_labels == label) & (predicted == value))
        for value in CATEGORY_LABELS.values()
    ]
    print(f"{category:<14} {counts[0]:>20} {counts[1]:>18}")

Độ chính xác: 29/35 (82.9%)
Luôn đoán lớp đông nhất: 22/35 (62.9%)

Nhãn thật           Đoán Nonfiction       Đoán Fiction
Nonfiction                       20                  2
Fiction                           4                  9


## 5. Thử dự đoán sách mới

Thay tiêu đề và mô tả trong ô bên dưới để thử một cuốn khác.

In [6]:
title = "The Clockmaker of London"
description = "A detective investigates a murder in a fictional Victorian town."

new_features = vectorizer.transform([f"{title} {description}"])
category_names = {label: category for category, label in CATEGORY_LABELS.items()}

if np.any(new_features):
    label = model.predict(new_features)[0]
    print(f"Danh mục dự đoán: {category_names[label]}")
else:
    print("Văn bản sách mới không có từ nào trong từ vựng huấn luyện.")

Danh mục dự đoán: Fiction
